In [1]:
# Ex1: Pollution, Jakob Eisenhauer

import pandas as pd
import numpy as np
from scipy.optimize import minimize

url = 'https://raw.githubusercontent.com/luchem/bern02/main/Labs/pollution_cleaneddata.csv'
data = pd.read_csv(url)

print(data.head())
print(data.columns)



   PREC  JANT  JULT  OVR65  POPN  EDUC  HOUS    DENS  NONW  WWDRK  POOR    HC  \
0  36.0  27.0  71.0    8.1  3.34  11.4  81.5  3243.0   8.8   42.6  11.7  21.0   
1  35.0  23.0  72.0   11.1  3.14  11.0  78.8  4281.0   3.5   50.7  14.4   8.0   
2  44.0  29.0  74.0   10.4  3.21   9.8  81.6  4260.0   0.8   39.4  12.4   6.0   
3  47.0  45.0  79.0    6.5  3.41  11.1  77.5  3125.0  27.1   50.2  20.6  18.0   
4  43.0  35.0  77.0    7.6  3.44   9.6  84.6  6441.0  24.4   43.7  14.3  43.0   

    NOX    SO@  HUMID      MORT  
0  15.0   59.0   59.0   921.870  
1  10.0   39.0   57.0   997.875  
2   6.0   33.0   54.0   962.354  
3   8.0   24.0   56.0   982.291  
4  38.0  206.0   55.0  1071.289  
Index(['PREC', 'JANT', 'JULT', 'OVR65', 'POPN', 'EDUC', 'HOUS', 'DENS', 'NONW',
       'WWDRK', 'POOR', 'HC', 'NOX', 'SO@', 'HUMID', 'MORT'],
      dtype='object')


In [2]:
POOR = data['POOR']
POOR = np.array(POOR)
MORT = data['MORT']
MORT = np.array(MORT)

In [3]:
def weights(valid_distances,w_max=2, w_min=1/2):
  '''
  d_min: maximum weight of w_max
  d_max: minimum weight of w_min
  w_max == w_min -> locally OLR
  inbetween: linear weight decrease
  '''
  d_min = min(valid_distances)
  d_max = max(valid_distances)
  if d_max == d_min:
    kNN_weights = np.ones(len(valid_distances))
  else:
    kNN_weights = w_max - (w_max-w_min)*((valid_distances-d_min)/(d_max-d_min))

  # mean_weight = 1
  normalisation = np.sum(kNN_weights)
  kNN_weights = kNN_weights/normalisation
  kNN_weights *= len(valid_distances)

  return kNN_weights



In [4]:
def objective(beta, x, y, kNN_weights):
  '''
  lossfunction for weighted linear regression
  '''
  beta0 = beta[0]
  beta1 = beta[1]

  # y_pred
  y_pred = beta1*x+beta0

  # residuals
  residuals_squared = (y-y_pred)**2

  # weighting
  weighted_residuals_squared_sum = np.sum(kNN_weights*residuals_squared)
  return weighted_residuals_squared_sum



In [5]:
def local_linear_regression(x,y,k,x0):
  '''
  Fits a linear model to the k nearest neighbours of x0[i] for all i
  '''

  y_preds = []
  SEs = []

  for i in range(len(x0)):
    #find kNN
    distances = np.abs(x-x0[i])
    sorted_indices = np.argsort(distances)
    kNN_indices = sorted_indices[:k]
    kNN_distances = distances[kNN_indices]

    kNN_x = x[kNN_indices]
    kNN_y = y[kNN_indices]

    #find weights
    kNN_weights = weights(kNN_distances)

    #optimize
    result = minimize(objective, x0=[0,0], args=(kNN_x, kNN_y, kNN_weights))
    beta = result.x
    y_pred_x0 = beta[0]+beta[1]*x0[i]

    #SE-estimation
    kNN_y_pred = beta[0]+beta[1]*kNN_x
    residuals = kNN_y-kNN_y_pred
    residuals_squared_weighted = kNN_weights*residuals**2
    weighted_sigma = np.sqrt(1/(k-2)*np.sum(residuals_squared_weighted))

    # for SE estimation not the x_mean but the weighted x_mean is needed
    # the 1/k term remains the same as in OLS as 1/sum(weights) = 1/k due to normalisation
    x_weighted_mean = np.sum(kNN_weights*kNN_x)/np.sum(kNN_weights)

    SE = weighted_sigma * np.sqrt( 1/k + (x0[i]-x_weighted_mean)**2/np.sum(kNN_weights*(kNN_x-x_weighted_mean)**2) )


    y_preds.append(y_pred_x0)
    SEs.append(SE)
  return y_preds, SEs






In [6]:
# Predictions for POOR = 10%, 18% and 25%

# k is set as approx. 1/3 of the dataset

k_data = len(POOR)*1/3
k_data = int(k_data)
print(k_data) # -> k = 20

x0_array = np.array([10,18,25])
y_preds, SEs = local_linear_regression(x=POOR,y=MORT,k=k_data,x0=x0_array)

print('-'*50)
for i in range(len(y_preds)):
  print(f'Predicted MORT for POOR = {x0_array[i]}%: {y_preds[i]:.1f}+-{SEs[i]:.1f}')

# Results

#Predicted MORT for POOR = 10%: 903.4+-16.8
#Predicted MORT for POOR = 18%: 951.7+-14.8
#Predicted MORT for POOR = 25%: 1006.4+-20.9

20
--------------------------------------------------
Predicted MORT for POOR = 10%: 903.4+-16.8
Predicted MORT for POOR = 18%: 951.7+-14.8
Predicted MORT for POOR = 25%: 1006.4+-20.9
